# 04 Photometrie-Stapel

**Eingabe**: kalibrierte FITS-Bilder (je Bild eine WCS) + `field_config.json` aus Notebook 03  
**Ausgabe**: `messtabelle.csv` — eine Zeile je Bild und Stern: Position, Fluss, Fehler, Himmel, Peak, Sättigung, Flags

Beispiel in diesem Notebook: **LCO-Archiv, δ Cep, Filter V, 0,4-m-Teleskop `0m4-11`** (öffentliche Daten, BANZAI-kalibriert).
Eigene Bilder: Pfade und Auswahl in Schritt 1 ändern.

Schritte: 1 Bilder wählen · 2 Sterne laden · 3 Photometrie · 4 Bildqualität (Flags) · 5 speichern

## Schritt 1 — Bilder und Pfade

`FRAMES_DIR` = Ordner mit den Bildern, `TELESCOPE` = optionaler Filter auf `TELESCOP` (hier nur das Teleskop `0m4-11`,
weil die Archivbilder von sechs Teleskopen mit anderer Pixelskala stammen). `FIELD_CONFIG` kommt aus NB03
(`VARPHOT_FRAME=<Referenzbild>` setzt dort Feld und Winkel aus dem Bild).

In [ ]:
import os, json
from pathlib import Path
import pandas as pd
from astropy.io import fits
from tqdm.auto import tqdm

from varphot import (CFG, run_batch, qc_frames, stars_from_field_config)
from varphot.observer import data_dir

FRAMES_DIR   = Path(os.environ.get("VARPHOT_FRAMES_DIR", "~/Downloads/00_run_software/sampledata_lco_dcep")).expanduser()
TELESCOPE    = os.environ.get("VARPHOT_TELESCOPE", "0m4-11")          # "" = alle
FIELD_CONFIG = Path(os.environ.get("VARPHOT_FIELD_CONFIG", "../data/fields/del_Cep_lco_0m4_field_config.json"))
TAG          = os.environ.get("VARPHOT_TAG", "delta_cep_lco_0m4_11")
RESULTS      = (data_dir() if data_dir().is_dir() else Path("../data")) / "results" / TAG
RESULTS.mkdir(parents=True, exist_ok=True)
OUT = RESULTS / "messtabelle.csv"

def _telescope(p):
    with fits.open(p) as h:
        hdr = h["SCI"].header if "SCI" in h else h[0].header
        return str(hdr.get("TELESCOP", ""))

alle = sorted(list(FRAMES_DIR.glob("*.fits.fz")) + list(FRAMES_DIR.glob("*.fits")) + list(FRAMES_DIR.glob("*.fit")))
FRAMES = [p for p in alle if not TELESCOPE or _telescope(p) == TELESCOPE]
print(f"{len(alle)} Bilder im Ordner, {len(FRAMES)} nach Teleskop-Filter '{TELESCOPE}'")
print("Ergebnisordner:", RESULTS)

## Schritt 2 — Sterne laden

Ziel, Vergleichssterne (Comps) und Check-Stern aus `field_config.json`. Gibt NB03 keinen Check-Stern her (kleines
Feld), wird er hier ausdrücklich aus den Comps bestimmt (`ROLLEN`): Er geht dann **nicht** in den Nullpunkt ein.

In [ ]:
record = json.loads(FIELD_CONFIG.read_text())
STARS = stars_from_field_config(record)
ROLLEN = {"g1": "check"}                  # Check-Stern: ein schwächerer Comp, der nicht in den Nullpunkt eingeht
STARS["role"] = [ROLLEN.get(i, r) for i, r in zip(STARS["id"], STARS["role"])]
STARS["id"] = STARS["id"].astype(str)
display(STARS.round(4))

## Schritt 3 — Stapellauf Aperturphotometrie

Je Bild: Positionen aus der WCS des Bildes, Apertur 10 px mit Himmelsring 16–24 px (bei 0,58″/px, Seeing ~2″),
Fluss, Fehler (CCD-Gleichung mit Gain/Ausleserauschen aus dem Header), Himmel, Peak, Zahl gesättigter Pixel.
Läuft parallel; Ergebnis wird zwischengespeichert (`skip_if_fresh`).

In [ ]:
from varphot import skip_if_fresh
CFG_RUN = {**CFG, "ap_r": 10, "ap_r_in": 16, "ap_r_out": 24}
RAW = RESULTS / "messtabelle_roh.csv"

if skip_if_fresh(RAW, FIELD_CONFIG, *[str(p) for p in FRAMES]):
    tab = pd.read_csv(RAW, keep_default_na=False, na_values=["nan", "NaN"], dtype={"star_id": str})
else:
    tab = run_batch(FRAMES, STARS, CFG_RUN, workers=4, progress=lambda it, total: tqdm(it, total=total))
    tab["star_id"] = tab["star_id"].astype(str)
    tab.to_csv(RAW, index=False)
print(f"{tab['file'].nunique()} Bilder × {tab['star_id'].nunique()} Sterne = {len(tab)} Messungen")

## Schritt 4 — Bildqualität (Flags)

Ein Bild ist **brauchbar**, wenn alle Comps genug Signal haben (`NO_SIGNAL`), die Transparenz stimmt
(`LOW_TRANSPARENCY`, dünne Wolken), das Seeing nicht schlecht ist (`BAD_SEEING`) und eine WCS vorhanden ist (`NO_WCS`).
Die Schwellen sind Startwerte (`snr_min`, `transp_min`, `fwhm_max_arcsec` in `CFG`).

In [ ]:
roles = dict(zip(STARS["id"], STARS["role"]))
tab = qc_frames(tab, roles, CFG_RUN)
fl = tab.drop_duplicates("file")[["file", "frame_flags", "transparency", "fwhm_arcsec"]]
print(f"{(fl['frame_flags'] == '').sum()} von {len(fl)} Bildern brauchbar")
print(fl["frame_flags"].replace("", "brauchbar").value_counts().to_string())

## Schritt 5 — Messtabelle speichern

In [ ]:
tab.to_csv(OUT, index=False)
print("gespeichert:", OUT)
display(tab[tab["star_id"] == "target"][["file", "jd_mid", "flux", "flux_err", "peak", "n_sat", "frame_flags"]].head())